In [78]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd


In [ ]:
# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent.parent
RESULTS_DIR = PROJECT_ROOT / "data" / "results"

sys.path.insert(0, str(PROJECT_ROOT))


In [13]:
with RAG_RESULTS_PATH.open(
    "r",
    encoding="utf-8"
) as file:
    rag_results = json.load(file)

print(
    "Saved RAG summaries:",
    len(rag_results)
)


# ------------------------------------------------------------
# Use one patient for the verification sanity test
# ------------------------------------------------------------

TEST_PATIENT_ID = "04df53ea-55c1-48d9-84a1-1f15c133b29b"

test_rag_summary = (
    rag_results[
        TEST_PATIENT_ID
    ]["summary"]
)

print("\n--- SAVED RAG SUMMARY ---\n")
print(test_rag_summary)

Saved RAG summaries: 50

--- SAVED RAG SUMMARY ---

- **01/01/2026:** Allan Victor Robinson, 67-year-old man with hypertension and osteoarthritis, no known drug allergies and no regular medications documented, presented to A&E with acute confusion after a minor fall. He was triage category 2, with GCS 12/15 and disorientation to time and place; initial BP was 170/95 and pulse 88. Urgent blood tests and CT head were arranged to exclude intracranial injury. Blood samples were sent for FNC, electrolytes and coagulation studies. CT head demonstrated a **subdural hygroma**, and admission to the Neurology Ward was planned. He remained haemodynamically stable with no external injuries.
- **01/01/2026:** Physiotherapy noted mild confusion, slight gait unsteadiness and no significant falls risk; daily balance and mobility therapy was planned. Occupational therapy identified mild fine-motor difficulties affecting buttons and utensil use, with therapy planned for ADLs and adaptive strategies. His

In [53]:
import re

from openai import OpenAI

In [64]:
ATOMIC_CLAIM_PROMPT = """
You are decomposing ONE source sentence from a generated clinical summary
into atomic claims.

Your only source is the sentence provided by the user.

TASK:
Extract every distinct factual proposition stated in the sentence as a
separate atomic claim.

An atomic claim contains exactly ONE fact that can independently receive
a verification label of SUPPORTED or UNSUPPORTED.

STRICT RULES:

1. Every claim must be explicitly stated in the supplied sentence.
2. Do not use outside knowledge.
3. Do not infer missing information.
4. Do not add, correct, interpret, or expand information.
5. Every claim must contain exactly ONE independently verifiable proposition.
6. Extract every explicitly stated factual proposition.
7. Each underlying fact should appear exactly once.
8. Do not return a broader compound claim if its individual facts have
   already been extracted.
9. Do not extract dates, patient names, or identifiers as standalone claims.
   Dates may remain as contextual information but are not themselves claims.
10. Do not extract contextual framing as a separate claim when it only
    establishes the setting for another clinical fact. For example,
    "During physiotherapy he walked five stairs" should produce a claim
    about walking five stairs, not a separate claim that physiotherapy occurred.

ATOMICITY RULE:

Ask of every candidate claim:

"Could one factual component of this claim be true while another factual
component is false?"

If YES, split the claim further.

Always split independently verifiable:
- demographic facts
- diagnoses
- symptoms
- findings
- measurements
- investigations
- procedures
- treatments
- medications
- medication dose
- medication frequency
- medication duration
- medication indication
- functional abilities
- clinical outcomes
- disposition decisions
- follow-up actions

SEMANTIC PRESERVATION:

- Preserve the meaning of the supplied sentence exactly.
- Do not transform wording into a different clinical meaning.
- Claims may be shorter than the source sentence but must be directly
  entailed by it.
- Do not create unnatural combinations of words from different parts
  of the sentence.

EXAMPLES:

SOURCE:
"Patient had hypertension and osteoarthritis."

CLAIMS:
"Patient had hypertension."
"Patient had osteoarthritis."


SOURCE:
"Atorvastatin 40 mg once daily was commenced as a 7-day course for cardiovascular risk."

CLAIMS:
"Atorvastatin was commenced."
"The atorvastatin dose was 40 mg."
"Atorvastatin was prescribed once daily."
"Atorvastatin was prescribed for 7 days."
"Atorvastatin was prescribed for cardiovascular risk."


SOURCE:
"The patient, a 67-year-old man, presented to A&E with acute confusion after a minor fall."

CLAIMS:
"The patient was 67 years old."
"The patient was male."
"The patient presented to A&E."
"The patient had acute confusion."
"The presentation followed a minor fall."


SOURCE:
"He was alert and oriented to person and place."

CLAIMS:
"He was alert."
"He was oriented to person."
"He was oriented to place."

FINAL CHECK:

A. Is every claim atomic?
B. Is every claim explicitly supported by this sentence?
C. Is any claim duplicated or overlapping?
D. Have all explicitly stated factual propositions been extracted?

Return valid JSON only.

Output format:
{
  "claims": [
    {
      "claim": "one atomic claim"
    }
  ]
}
"""

In [65]:
def split_summary_into_source_units(summary: str):

    units = []

    for line in summary.splitlines():

        line = line.strip()

        if not line:
            continue

        # Remove markdown bullet marker if present
        cleaned = re.sub(
            r"^-\s*",
            "",
            line
        ).strip()

        # Split into actual sentences.
        # Split after . ! ? when followed by whitespace and the next
        # sentence begins with a capital letter or markdown formatting.
        sentences = re.split(
            r'(?<=[.!?])\s+(?=(?:\*\*)?[A-Z])',
            cleaned
        )

        for sentence in sentences:

            sentence = sentence.strip()

            if sentence:
                units.append(sentence)

    return units

In [66]:
source_units = split_summary_into_source_units(
    test_rag_summary
)

print(
    "Source units:",
    len(source_units)
)

for i, unit in enumerate(
    source_units,
    start=1
):
    print(f"\n{i}. {unit}")

Source units: 23

1. **01/01/2026:** Allan Victor Robinson, 67-year-old man with hypertension and osteoarthritis, no known drug allergies and no regular medications documented, presented to A&E with acute confusion after a minor fall.

2. He was triage category 2, with GCS 12/15 and disorientation to time and place; initial BP was 170/95 and pulse 88.

3. Urgent blood tests and CT head were arranged to exclude intracranial injury.

4. Blood samples were sent for FNC, electrolytes and coagulation studies.

5. CT head demonstrated a **subdural hygroma**, and admission to the Neurology Ward was planned.

6. He remained haemodynamically stable with no external injuries.

7. **01/01/2026:** Physiotherapy noted mild confusion, slight gait unsteadiness and no significant falls risk; daily balance and mobility therapy was planned.

8. Occupational therapy identified mild fine-motor difficulties affecting buttons and utensil use, with therapy planned for ADLs and adaptive strategies.

9. His ne

In [161]:
from openai import OpenAI

openai_client = OpenAI()

In [162]:
from openai import OpenAI

openai_client = OpenAI()

EVALUATOR_MODEL = "gpt-5.4-mini"
LLM_MODEL = "gpt-5.6-luna"

In [68]:
def extract_claims_from_source_unit(source_unit):

    response = openai_client.chat.completions.create(
        model=EVALUATOR_MODEL,
        messages=[
            {
                "role": "system",
                "content": ATOMIC_CLAIM_PROMPT
            },
            {
                "role": "user",
                "content": source_unit
            }
        ],
        response_format={
            "type": "json_object"
        }
    )

    result = json.loads(
        response.choices[0].message.content
    )

    return result["claims"]

In [ ]:
def extract_atomic_claims(summary):

    source_units = split_summary_into_source_units(
        summary
    )

    all_claims = []

    for source_id, source_unit in enumerate(
        source_units,
        start=1
    ):

        claims = extract_claims_from_source_unit(
            source_unit
        )

        for item in claims:

            if "claim" not in item:
                print(
                    "Malformed claim item:",
                    item
                )
                continue

            all_claims.append(
                {
                    "claim": item["claim"],
                    "source_id": source_id,
                    "source_sentence": source_unit
                }
        )

    return all_claims

In [70]:
test_claims = extract_atomic_claims(
    test_rag_summary
)

print(
    "Atomic claims:",
    len(test_claims)
)

for i, item in enumerate(
    test_claims,
    start=1
):
    print(f"\n{i}. CLAIM:")
    print(item["claim"])

    print("SOURCE ID:")
    print(item["source_id"])

    print("SOURCE:")
    print(item["source_sentence"])

Atomic claims: 89

1. CLAIM:
The patient was 67 years old.
SOURCE ID:
1
SOURCE:
**01/01/2026:** Allan Victor Robinson, 67-year-old man with hypertension and osteoarthritis, no known drug allergies and no regular medications documented, presented to A&E with acute confusion after a minor fall.

2. CLAIM:
The patient was male.
SOURCE ID:
1
SOURCE:
**01/01/2026:** Allan Victor Robinson, 67-year-old man with hypertension and osteoarthritis, no known drug allergies and no regular medications documented, presented to A&E with acute confusion after a minor fall.

3. CLAIM:
The patient had hypertension.
SOURCE ID:
1
SOURCE:
**01/01/2026:** Allan Victor Robinson, 67-year-old man with hypertension and osteoarthritis, no known drug allergies and no regular medications documented, presented to A&E with acute confusion after a minor fall.

4. CLAIM:
The patient had osteoarthritis.
SOURCE ID:
1
SOURCE:
**01/01/2026:** Allan Victor Robinson, 67-year-old man with hypertension and osteoarthritis, no kn

In [72]:
import numpy as np
from sentence_transformers import SentenceTransformer


In [73]:
bge_model = SentenceTransformer(
    "BAAI/bge-base-en-v1.5"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [74]:
def split_note_into_sections(note_text):
    """
    Split a clinical note using explicit section headings already
    present in the note.

    Returns:
        list of dictionaries:
        [
            {
                "section_name": "...",
                "chunk_text": "..."
            },
            ...
        ]
    """

    text = str(note_text).strip()

    if not text:
        return []

    section_names = [
        "Chief Complaint",
        "Presenting Complaint",
        "History of Present Illness",
        "HPI",
        "Past Medical History",
        "PMH",
        "Past Surgical History",
        "PSH",
        "Medications",
        "Current Medications",
        "Allergies",
        "Family History",
        "Social History",
        "Review of Systems",
        "ROS",
        "Physical Examination",
        "Physical Exam",
        "Examination",
        "Vital Signs",
        "Vitals",
        "Investigations",
        "Laboratory Results",
        "Labs",
        "Imaging",
        "Assessment",
        "Impression",
        "Diagnosis",
        "Diagnoses",
        "Plan",
        "Assessment and Plan",
        "Treatment",
        "Hospital Course",
        "Clinical Course",
        "Discharge Plan",
        "Follow Up",
        "Follow-Up",
    ]

    heading_pattern = "|".join(
        re.escape(name)
        for name in sorted(
            section_names,
            key=len,
            reverse=True
        )
    )

    pattern = re.compile(
        rf"(?im)^[ \t]*(?P<heading>{heading_pattern})"
        rf"[ \t]*(?::|-)?[ \t]*$"
    )

    matches = list(pattern.finditer(text))

    # If no recognized headings exist, keep the full note.
    if not matches:
        return [
            {
                "section_name": "Unsectioned",
                "chunk_text": text,
            }
        ]

    sections = []

    # Preserve text before the first heading.
    prefix = text[:matches[0].start()].strip()

    if prefix:
        sections.append(
            {
                "section_name": "Preamble",
                "chunk_text": prefix,
            }
        )

    # Extract each section.
    for i, match in enumerate(matches):

        section_name = match.group("heading").strip()

        content_start = match.end()

        if i + 1 < len(matches):
            content_end = matches[i + 1].start()
        else:
            content_end = len(text)

        content = text[
            content_start:content_end
        ].strip()

        if content:
            sections.append(
                {
                    "section_name": section_name,
                    "chunk_text": (
                        f"{section_name}\n{content}"
                    ),
                }
            )

    return sections

In [79]:
# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent.parent
DATA_DIR = PROJECT_ROOT / "data"

sys.path.insert(0, str(PROJECT_ROOT))


# ------------------------------------------------------------
# Load clinical notes
# ------------------------------------------------------------

notes = pd.read_csv(
    DATA_DIR / "raw" / "clinical_notes.csv"
)

print("Original notes:", len(notes))
print("Columns:", notes.columns.tolist())

Original notes: 1602
Columns: ['ingest_timestamp', 'clinical_note_id', 'clean_note_text', 'creation_timestamp', 'updt_dt_tm', 'note_subject', 'note_type', 'admission_id', 'person_id']


In [80]:
# ------------------------------------------------------------
# Remove broken notes
# ------------------------------------------------------------

notes_clean = notes[
    notes["clean_note_text"]
    .astype(str)
    .str.strip()
    != "#NAME?"
].copy()

print(
    "After #NAME? removal:",
    len(notes_clean)
)


# ------------------------------------------------------------
# Deduplicate repeated note content per patient
# ------------------------------------------------------------

notes_dedup = (
    notes_clean
    .sort_values(
        [
            "person_id",
            "creation_timestamp",
        ]
    )
    .drop_duplicates(
        subset=[
            "person_id",
            "clean_note_text",
        ],
        keep="first",
    )
    .reset_index(drop=True)
)

print(
    "After deduplication:",
    len(notes_dedup)
)

print(
    "Patients:",
    notes_dedup["person_id"].nunique()
)

After #NAME? removal: 1595
After deduplication: 1103
Patients: 50


In [81]:
section_records = []

for _, row in notes_dedup.iterrows():
    sections = split_note_into_sections(
        row["clean_note_text"]
    )

    for section in sections:
        section_records.append({
            "person_id": row["person_id"],
            "creation_timestamp": row["creation_timestamp"],
            "section_name": section["section_name"],
            "chunk_text": section["chunk_text"],
        })


section_chunks = pd.DataFrame(
    section_records
)

section_chunks = (
    section_chunks
    .sort_values(
        [
            "person_id",
            "creation_timestamp",
        ]
    )
    .reset_index(drop=True)
)

section_chunks["chunk_id"] = range(
    len(section_chunks)
)

print(
    "Total section chunks:",
    len(section_chunks)
)

print(
    "Patients:",
    section_chunks["person_id"].nunique()
)

display(section_chunks.head(10))

Total section chunks: 2771
Patients: 50


,person_id,creation_timestamp,section_name,chunk_text,chunk_id
0,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 08:45,Unsectioned,"- Patient: Tomos Ellis, 15-year-old male, pres...",0
1,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 09:10,Unsectioned,"Patient: Tomos Ellis, 15-year-old male, presen...",1
2,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 09:25,Unsectioned,"Patient name: Tomos Ellis, 15-year-old male. N...",2
3,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 10:00,Unsectioned,"Reviewed abdominal X-ray on 2026-01-07, which ...",3
4,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 10:30,Preamble,Patient\nTomos Ellis\n\nAge\n15\n\nSex\nMale\n...,4
5,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 10:30,Presenting Complaint,Presenting Complaint\nAbdominal pain\n\nHistor...,5
6,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 10:30,Past Medical History,Past Medical History\nNil,6
7,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 10:30,Medications,Medications\nNil,7
8,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 10:30,Allergies,Allergies\nNKDA,8
9,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 10:30,Social History,Social History\nLives at home with parents. At...,9


In [82]:
patient_evidence = (
    section_chunks[
        section_chunks["person_id"] == TEST_PATIENT_ID
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Test patient evidence chunks:",
    len(patient_evidence)
)

display(
    patient_evidence[
        [
            "chunk_id",
            "creation_timestamp",
            "section_name",
            "chunk_text",
        ]
    ]
)

Test patient evidence chunks: 71


,chunk_id,creation_timestamp,section_name,chunk_text
0,43,01/01/2026 07:30,Unsectioned,"67M, Allan Victor Robinson (DOB: 1956-03-15, N..."
1,44,01/01/2026 08:00,Unsectioned,67M. GCS 12/15 - disoriented to time/place. BP...
2,45,01/01/2026 08:30,Unsectioned,"- Patient: Allan Victor R obinson, 67-year-old..."
3,46,01/01/2026 10:15,Unsectioned,"10:15, 01/01/26 - Patient Allan Victor Robinso..."
4,47,01/01/2026 10:45,Preamble,Patient\nAllan Victor Robinson\n\nAge\n67\n\nS...
...,...,...,...,...
66,109,06/01/2026 15:00,Unsectioned,Pt ed session by N. Hopkins. Info on dizziness...
67,110,07/01/2026 08:30,Unsectioned,Patient clinically stable. No residual confusi...
68,111,07/01/2026 10:00,Unsectioned,Patient Allan Victor Robinson seen for final p...
69,112,07/01/2026 14:00,Unsectioned,Disch summary completed by Nurse Hopkins. Incl...


In [83]:
evidence_texts = (
    patient_evidence["chunk_text"]
    .astype(str)
    .tolist()
)

evidence_embeddings = bge_model.encode(
    evidence_texts,
    normalize_embeddings=True,
    show_progress_bar=False
)

print(
    "Evidence embedding shape:",
    evidence_embeddings.shape
)

Evidence embedding shape: (71, 768)


In [94]:
VERIFICATION_TOP_K = 10

def retrieve_evidence_for_claim(
    claim,
    source_sentence,
    patient_evidence,
    evidence_embeddings,
    top_k=VERIFICATION_TOP_K
):

    retrieval_query = (
        f"Claim: {claim}\n"
        f"Context: {source_sentence}"
    )

    claim_embedding = bge_model.encode(
        [retrieval_query],
        normalize_embeddings=True,
        show_progress_bar=False
    )[0]

    similarities = (
        evidence_embeddings
        @ claim_embedding
    )

    top_indices = np.argsort(
        similarities
    )[::-1][:top_k]

    retrieved = []

    for rank, idx in enumerate(
        top_indices,
        start=1
    ):
        row = patient_evidence.iloc[idx]

        retrieved.append({
            "rank": rank,
            "chunk_id": row["chunk_id"],
            "creation_timestamp": str(
                row["creation_timestamp"]
            ),
            "section_name": row["section_name"],
            "text": row["chunk_text"],
            "similarity": float(
                similarities[idx]
            ),
        })

    return retrieved

In [95]:
claim_84 = test_claims[83]

print("CLAIM:")
print(claim_84["claim"])

print("\nSOURCE SENTENCE:")
print(claim_84["source_sentence"])

CLAIM:
GCS was 15/15.

SOURCE SENTENCE:
Final physiotherapy confirmed independence with walking, five stairs using a handrail and obstacle navigation; GCS was 15/15.


In [96]:
claim_84_evidence = retrieve_evidence_for_claim(
    claim=claim_84["claim"],
    source_sentence=claim_84["source_sentence"],
    patient_evidence=patient_evidence,
    evidence_embeddings=evidence_embeddings
)

for item in claim_84_evidence:
    print(f"\n--- RANK {item['rank']} ---")
    print("Similarity:", round(item["similarity"], 4))
    print("Date:", item["creation_timestamp"])
    print("TEXT:")
    print(item["text"])


--- RANK 1 ---
Similarity: 0.7682
Date: 01/01/2026 08:30
TEXT:
- Patient: Allan Victor R obinson, 67-year-old male, NHS number 680034032. 
 - Date of event: 01/01/26, time: 28:30.
 - Event conducted by: Nurse Kay Claire Williamson. 
 - Details of event: Initial blood draws completed to investigate potential intracranial injury; patient prepared for CT head scan. 
 - Blood samples sent for: FNC, electrolytes, coagulation profile. 
 - Patient escorted to radiology for CT scan. 
 - Clinical context: Patient presented with acute confusion following a minor fall and was assigned triage category 2. GCS score was noted as 12/15 with disorientation to time and place. Past medical history includes HTN and osteoarthritis. The CT scan was ordered to investigate potential intracranial injury, specifically subdural hygroma.
Nurse Kay Claire Williamson 
NMC number: 07P9395S

--- RANK 2 ---
Similarity: 0.7485
Date: 05/01/2026 10:30
TEXT:
Date: 2026-01-05
Time: 10:30

Patient seen for final physiothe

In [85]:
test_claim = "His GCS was 12/15."

retrieved_evidence = retrieve_evidence_for_claim(
    claim=test_claim,
    patient_evidence=patient_evidence,
    evidence_embeddings=evidence_embeddings
)

print("CLAIM:")
print(test_claim)

for item in retrieved_evidence:
    print(
        f"\n--- RANK {item['rank']} ---"
    )
    print(
        "Similarity:",
        round(
            item["similarity"],
            4
        )
    )
    print(
        "Chunk ID:",
        item["chunk_id"]
    )
    print(
        "Date:",
        item["creation_timestamp"]
    )
    print(
        "Section:",
        item["section_name"]
    )
    print(
        "TEXT:"
    )
    print(
        item["text"]
    )

CLAIM:
His GCS was 12/15.

--- RANK 1 ---
Similarity: 0.5484
Chunk ID: 45
Date: 01/01/2026 08:30
Section: Unsectioned
TEXT:
- Patient: Allan Victor R obinson, 67-year-old male, NHS number 680034032. 
 - Date of event: 01/01/26, time: 28:30.
 - Event conducted by: Nurse Kay Claire Williamson. 
 - Details of event: Initial blood draws completed to investigate potential intracranial injury; patient prepared for CT head scan. 
 - Blood samples sent for: FNC, electrolytes, coagulation profile. 
 - Patient escorted to radiology for CT scan. 
 - Clinical context: Patient presented with acute confusion following a minor fall and was assigned triage category 2. GCS score was noted as 12/15 with disorientation to time and place. Past medical history includes HTN and osteoarthritis. The CT scan was ordered to investigate potential intracranial injury, specifically subdural hygroma.
Nurse Kay Claire Williamson 
NMC number: 07P9395S

--- RANK 2 ---
Similarity: 0.5362
Chunk ID: 44
Date: 01/01/2026 0

In [118]:
VERIFICATION_PROMPT = """
You are verifying one atomic clinical claim against retrieved evidence
from the patient's original clinical record.

TASK:
Determine whether the claim is supported by the evidence provided.

LABELS:

SUPPORTED:
The evidence directly supports the full factual meaning of the claim.

PARTIALLY_SUPPORTED:
Some part of the claim is supported, but another factual component is
missing, uncertain, or contradicted.

UNSUPPORTED:
The evidence does not support the claim, or the evidence contradicts it.

RULES:

1. Use only the evidence provided.
2. Do not use outside medical knowledge.
3. Do not infer facts that are not explicitly supported.
4. Semantic equivalence is allowed; exact wording is not required.
5. Do not treat a high semantic similarity score as proof.
6. If the evidence explicitly contradicts the claim, label UNSUPPORTED.
7. If the full atomic claim is supported, label SUPPORTED.
8. Respect temporal context. Clinical states may change over time.
   Earlier or later conflicting findings do not automatically contradict
   a claim if the claim is supported at the relevant time point.
9. When the claim is derived from a temporally specific source sentence,
   prioritize evidence from the same clinical time period or event.
10. Do not downgrade a claim merely because the patient's condition was
    different earlier or later in the admission.
11. SUMMARY CONTEXT is provided only to identify the intended clinical
    time point or event. It is NOT evidence and must never be used to
    support the claim.
12. Base the label and explanation only on the retrieved original
    clinical evidence. In the reason, do not cite the summary context
    as proof.
13. Interpret the atomic claim within the time point, event, or clinical
    state specified by SUMMARY CONTEXT.

    For example, if SUMMARY CONTEXT places a claim on 07/01/2026,
    judge whether the original evidence supports that claim on or around
    07/01/2026. Do not interpret the atomic claim as applying to the
    patient's entire admission.
14. Findings from other time points are not contradictions when the
    patient's clinical state could reasonably change over time.
15. SUMMARY CONTEXT may define the scope of the claim, but it may not
    be used as evidence that the claim is true. Supporting evidence must
    still come exclusively from the retrieved original clinical record.

Return valid JSON only.

Output format:
{
  "label": "SUPPORTED | PARTIALLY_SUPPORTED | UNSUPPORTED",
  "reason": "brief explanation",
  "supporting_evidence_ranks": [1]
}
"""

In [123]:
def verify_claim(
    claim,
    source_sentence,
    retrieved_evidence
):

    evidence_text = "\n\n".join(
        [
            (
                f"EVIDENCE {item['rank']}:\n"
                f"{item['text']}"
            )
            for item in retrieved_evidence
        ]
    )

    user_content = (
        f"ATOMIC CLAIM:\n{claim}\n\n"
        f"SUMMARY CONTEXT:\n{source_sentence}\n\n"
        f"{evidence_text}"
    )

    response = openai_client.chat.completions.create(
        model=EVALUATOR_MODEL,
        messages=[
            {
                "role": "system",
                "content": VERIFICATION_PROMPT
            },
            {
                "role": "user",
                "content": user_content
            }
        ],
        response_format={
            "type": "json_object"
        }
    )

    return json.loads(
        response.choices[0].message.content
    )

In [120]:
claim_84_verification = verify_claim(
    claim_84["claim"],
    source_sentence=claim_84["source_sentence"],
    retrieved_evidence=claim_84_evidence
)

print(
    json.dumps(
        claim_84_verification,
        indent=2
    )
)

{
  "label": "SUPPORTED",
  "reason": "Evidence 3 explicitly states the final physio session on 07/01/26 showed the patient was neurologically intact with GCS 15/15.",
  "supporting_evidence_ranks": [
    3
  ]
}


In [121]:
for claim_num in [55, 80, 82]:

    item = test_claims[claim_num - 1]

    evidence = retrieve_evidence_for_claim(
        claim=item["claim"],
        source_sentence=item["source_sentence"],
        patient_evidence=patient_evidence,
        evidence_embeddings=evidence_embeddings
    )

    result = verify_claim(
        claim=item["claim"],
        source_sentence=item["source_sentence"],
        retrieved_evidence=evidence
    )

    print("\nCLAIM", claim_num)
    print("Claim:", item["claim"])
    print("Context:", item["source_sentence"])
    print(json.dumps(result, indent=2))


CLAIM 55
Claim: Mild residual confusion remained.
Context: Mobility and balance were improving; mild residual confusion remained, although he was alert and oriented to person and place.
{
  "label": "SUPPORTED",
  "reason": "The evidence shows the patient had improving confusion with residual mild confusion persisting on 01/01/26 and 01/02/26; later notes show resolution, but at the relevant time the claim is directly supported.",
  "supporting_evidence_ranks": [
    5,
    8
  ]
}

CLAIM 80
Claim: The patient had no neurological deficits.
Context: **07/01/2026:** The patient was clinically stable without residual confusion or neurological deficits.
{
  "label": "SUPPORTED",
  "reason": "The record explicitly states on 07/01/26 that the patient was 'neurologically intact' with 'no confusion,' and earlier the consultant note says 'No residual confusion or neurological deficits.'",
  "supporting_evidence_ranks": [
    6,
    2
  ]
}

CLAIM 82
Claim: Final physiotherapy confirmed indepen

In [126]:
item = test_claims[10]

print("Testing claim:")
print(item["claim"])

gcs_evidence = retrieve_evidence_for_claim(
    claim=item["claim"],
    source_sentence=item["source_sentence"],
    patient_evidence=patient_evidence,
    evidence_embeddings=evidence_embeddings
)

gcs_verification = verify_claim(
    claim=item["claim"],
    source_sentence=item["source_sentence"],
    retrieved_evidence=gcs_evidence
)

print(
    json.dumps(
        gcs_verification,
        indent=2
    )
)

Testing claim:
His GCS was 12/15.
{
  "label": "SUPPORTED",
  "reason": "The record explicitly states that the patient had an initial GCS of 12/15 during triage and on presentation, with disorientation to time and place.",
  "supporting_evidence_ranks": [
    2
  ]
}


In [124]:
verification_result = verify_claim(
    claim=item["claim"],
    source_sentence=item["source_sentence"],
    retrieved_evidence=retrieved_evidence
)

print(
    json.dumps(
        verification_result,
        indent=2
    )
)

{
  "label": "UNSUPPORTED",
  "reason": "The evidence mentions only a physio referral for mobilization assessment and an unsteady gait; it does not document any final physiotherapy assessment or confirm independence with five stairs using a handrail.",
  "supporting_evidence_ranks": [
    3,
    5
  ]
}


In [127]:
def verify_all_claims(
    claims,
    patient_evidence,
    evidence_embeddings
):
    results = []

    for i, item in enumerate(
        claims,
        start=1
    ):
        claim_text = item["claim"]

        retrieved_evidence = retrieve_evidence_for_claim(
            claim=claim_text,
            source_sentence=item["source_sentence"],
            patient_evidence=patient_evidence,
            evidence_embeddings=evidence_embeddings
        )

        verification = verify_claim(
            claim=claim_text,
            source_sentence=item["source_sentence"],
            retrieved_evidence=retrieved_evidence
        )

        results.append(
            {
                "claim_id": i,
                "claim": claim_text,
                "source_id": item["source_id"],
                "source_sentence": item["source_sentence"],
                "label": verification["label"],
                "reason": verification["reason"],
                "supporting_evidence_ranks": verification[
                    "supporting_evidence_ranks"
                ],
                "retrieved_evidence": retrieved_evidence,
            }
        )

    return results

In [128]:
test_verification_results = verify_all_claims(
    claims=test_claims,
    patient_evidence=patient_evidence,
    evidence_embeddings=evidence_embeddings
)

print(
    "Verified claims:",
    len(test_verification_results)
)

Verified claims: 89


In [129]:
from collections import Counter

label_counts = Counter(
    item["label"]
    for item in test_verification_results
)

print(label_counts)

Counter({'SUPPORTED': 89})


In [130]:
for item in test_verification_results:
    if item["label"] != "SUPPORTED":

        print("\nCLAIM ID:", item["claim_id"])
        print("CLAIM:", item["claim"])
        print("LABEL:", item["label"])
        print("REASON:", item["reason"])

In [131]:
false_claims = [
    {
        "claim": "His GCS was 10/15.",
        "source_sentence": "This is a synthetic negative-control claim."
    },
    {
        "claim": "The patient was 45 years old.",
        "source_sentence": "This is a synthetic negative-control claim."
    },
    {
        "claim": "The patient was prescribed insulin.",
        "source_sentence": "This is a synthetic negative-control claim."
    }
]

In [132]:
for i, item in enumerate(false_claims, start=1):

    evidence = retrieve_evidence_for_claim(
        claim=item["claim"],
        source_sentence=item["source_sentence"],
        patient_evidence=patient_evidence,
        evidence_embeddings=evidence_embeddings
    )

    result = verify_claim(
        claim=item["claim"],
        source_sentence=item["source_sentence"],
        retrieved_evidence=evidence
    )

    print(f"\nFALSE CLAIM {i}")
    print("Claim:", item["claim"])
    print(json.dumps(result, indent=2))


FALSE CLAIM 1
Claim: His GCS was 10/15.
{
  "label": "UNSUPPORTED",
  "reason": "The records consistently document a GCS of 12/15 initially, with later improvement to 13/15 and then 15/15. There is no evidence that it was 10/15 at the relevant time.",
  "supporting_evidence_ranks": [
    1,
    2,
    5,
    7,
    9
  ]
}

FALSE CLAIM 2
Claim: The patient was 45 years old.
{
  "label": "UNSUPPORTED",
  "reason": "The evidence identifies the patient as 67 years old, not 45. No evidence supports age 45.",
  "supporting_evidence_ranks": [
    1,
    2,
    5,
    6,
    10
  ]
}

FALSE CLAIM 3
Claim: The patient was prescribed insulin.
{
  "label": "UNSUPPORTED",
  "reason": "The evidence does not mention insulin being prescribed. One note explicitly says 'No meds.'",
  "supporting_evidence_ranks": [
    4,
    9
  ]
}


In [133]:
def build_verified_summary_for_patient(
    original_summary,
    verification_results
):
    non_supported = [
        item
        for item in verification_results
        if item["label"] != "SUPPORTED"
    ]

    # If every claim is supported, keep the original summary unchanged.
    if len(non_supported) == 0:
        return {
            "revision_needed": False,
            "final_summary": original_summary,
            "non_supported_claims": []
        }

    return {
        "revision_needed": True,
        "final_summary": None,
        "non_supported_claims": non_supported
    }

In [134]:
test_revision_check = build_verified_summary_for_patient(
    original_summary=test_rag_summary,
    verification_results=test_verification_results
)

print(
    "Revision needed:",
    test_revision_check["revision_needed"]
)

print(
    "Non-supported claims:",
    len(test_revision_check["non_supported_claims"])
)

Revision needed: False
Non-supported claims: 0


In [135]:
test_verified_summary = test_revision_check[
    "final_summary"
]

print(
    "Summary unchanged:",
    test_verified_summary == test_rag_summary
)

Summary unchanged: True


In [136]:
REVISION_PROMPT = """
You are revising a longitudinal clinical summary after factual verification.

You will receive:
1. The original RAG-generated clinical summary.
2. A list of claims that were identified as PARTIALLY_SUPPORTED or UNSUPPORTED.
3. Verification explanations and supporting original clinical evidence.

TASK:
Revise the original summary only where necessary.

RULES:
1. Preserve all fully supported information unchanged as much as possible.
2. Remove claims labeled UNSUPPORTED.
3. For PARTIALLY_SUPPORTED claims, retain only the portion supported by
   the original clinical evidence.
4. Do not introduce any new clinical facts.
5. Do not infer missing information.
6. Preserve the original chronology and overall structure.
7. Make the smallest possible edits needed to produce a fully
   evidence-supported summary.
8. Return only the revised clinical summary.
"""

In [138]:
from src.llm.llm import LLM_MODEL


def revise_summary_with_verification(
    original_summary,
    verification_results
):
    non_supported = [
        item
        for item in verification_results
        if item["label"] != "SUPPORTED"
    ]

    # Nothing to revise
    if len(non_supported) == 0:
        return original_summary

    issues_text = "\n\n".join(
        [
            (
                f"CLAIM: {item['claim']}\n"
                f"LABEL: {item['label']}\n"
                f"REASON: {item['reason']}"
            )
            for item in non_supported
        ]
    )

    user_content = (
        f"ORIGINAL SUMMARY:\n{original_summary}\n\n"
        f"VERIFICATION ISSUES:\n{issues_text}"
    )

    response = openai_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {
                "role": "system",
                "content": REVISION_PROMPT
            },
            {
                "role": "user",
                "content": user_content
            }
        ]
    )

    return response.choices[0].message.content.strip()

In [139]:
test_verified_summary = revise_summary_with_verification(
    original_summary=test_rag_summary,
    verification_results=test_verification_results
)

print(
    "Summary unchanged:",
    test_verified_summary == test_rag_summary
)

Summary unchanged: True


In [140]:
print("===== ORIGINAL RAG SUMMARY =====\n")
print(test_rag_summary)

print("\n\n===== WORKFLOW 4 VERIFIED SUMMARY =====\n")
print(test_verified_summary)

===== ORIGINAL RAG SUMMARY =====

- **01/01/2026:** Allan Victor Robinson, 67-year-old man with hypertension and osteoarthritis, no known drug allergies and no regular medications documented, presented to A&E with acute confusion after a minor fall. He was triage category 2, with GCS 12/15 and disorientation to time and place; initial BP was 170/95 and pulse 88. Urgent blood tests and CT head were arranged to exclude intracranial injury. Blood samples were sent for FNC, electrolytes and coagulation studies. CT head demonstrated a **subdural hygroma**, and admission to the Neurology Ward was planned. He remained haemodynamically stable with no external injuries.
- **01/01/2026:** Physiotherapy noted mild confusion, slight gait unsteadiness and no significant falls risk; daily balance and mobility therapy was planned. Occupational therapy identified mild fine-motor difficulties affecting buttons and utensil use, with therapy planned for ADLs and adaptive strategies. His next of kin was u

In [141]:
print("\nOriginal characters:", len(test_rag_summary))
print("Verified characters:", len(test_verified_summary))
print("Exactly identical:", test_rag_summary == test_verified_summary)


Original characters: 2674
Verified characters: 2674
Exactly identical: True


In [142]:
with open(RAG_RESULTS_PATH, "r") as f:
    rag_results = json.load(f)

print("RAG patients:", len(rag_results))

RAG patients: 50


In [147]:
rag_results_test = list(rag_results.items())[:2]

In [148]:
all_verification_results_test = []
all_verified_summaries_test = []

for patient_idx, (person_id, rag_item) in enumerate(
    rag_results_test,
    start=1
):
    original_summary = rag_item["summary"]

    print(
        f"\nProcessing patient "
        f"{patient_idx}/{len(rag_results_test)}"
    )
    print("Person ID:", person_id)

    patient_evidence = (
        section_chunks[
            section_chunks["person_id"] == person_id
        ]
        .copy()
        .reset_index(drop=True)
    )

    evidence_texts = (
        patient_evidence["chunk_text"]
        .astype(str)
        .tolist()
    )

    evidence_embeddings = bge_model.encode(
        evidence_texts,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    patient_claims = extract_atomic_claims(
        original_summary
    )

    verification_results = verify_all_claims(
        claims=patient_claims,
        patient_evidence=patient_evidence,
        evidence_embeddings=evidence_embeddings
    )

    verified_summary = revise_summary_with_verification(
        original_summary=original_summary,
        verification_results=verification_results
    )

    label_counts = Counter(
        item["label"]
        for item in verification_results
    )

    all_verification_results_test.append(
        {
            "person_id": person_id,
            "claim_count": len(patient_claims),
            "label_counts": dict(label_counts),
            "verification_results": verification_results,
        }
    )

    all_verified_summaries_test.append(
        {
            "person_id": person_id,
            "original_rag_summary": original_summary,
            "verified_summary": verified_summary,
            "summary_changed": (
                verified_summary != original_summary
            ),
        }
    )

    print("Claims:", len(patient_claims))
    print("Labels:", label_counts)
    print(
        "Summary changed:",
        verified_summary != original_summary
    )


Processing patient 1/2
Person ID: 028998ee-babc-4096-9b28-001bc2f9a84e
Claims: 72
Labels: Counter({'SUPPORTED': 72})
Summary changed: False

Processing patient 2/2
Person ID: 04df53ea-55c1-48d9-84a1-1f15c133b29b
Claims: 89
Labels: Counter({'SUPPORTED': 89})
Summary changed: False


In [149]:
from pathlib import Path
import json

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

ATOMIC_CLAIMS_PATH = (
    PROCESSED_DIR
    / "rag_atomic_claims.json"
)

print(ATOMIC_CLAIMS_PATH)

/Users/pallavi_chandanshive/projects/clinical-summarization-eval/data/processed/rag_atomic_claims.json


In [153]:
import json

# Load any previously saved progress
if ATOMIC_CLAIMS_PATH.exists():
    with open(ATOMIC_CLAIMS_PATH, "r") as f:
        all_atomic_claims = json.load(f)
else:
    all_atomic_claims = {}

print(
    "Already completed:",
    len(all_atomic_claims)
)

for patient_idx, (person_id, rag_item) in enumerate(
    rag_results.items(),
    start=1
):
    # Skip already completed patients
    if person_id in all_atomic_claims:
        print(
            f"Skipping patient {patient_idx}/{len(rag_results)} "
            f"(already saved)"
        )
        continue

    original_summary = rag_item["summary"]

    print(
        f"\nProcessing patient "
        f"{patient_idx}/{len(rag_results)}"
    )
    print("Person ID:", person_id)

    patient_claims = extract_atomic_claims(
        original_summary
    )

    all_atomic_claims[person_id] = {
        "person_id": person_id,
        "claim_count": len(patient_claims),
        "claims": patient_claims,
    }

    # Save immediately after each patient
    with open(
        ATOMIC_CLAIMS_PATH,
        "w"
    ) as f:
        json.dump(
            all_atomic_claims,
            f,
            indent=2
        )

    print(
        "Claims:",
        len(patient_claims)
    )
    print(
        "Checkpoint saved."
    )

Already completed: 27
Skipping patient 1/50 (already saved)
Skipping patient 2/50 (already saved)
Skipping patient 3/50 (already saved)
Skipping patient 4/50 (already saved)
Skipping patient 5/50 (already saved)
Skipping patient 6/50 (already saved)
Skipping patient 7/50 (already saved)
Skipping patient 8/50 (already saved)
Skipping patient 9/50 (already saved)
Skipping patient 10/50 (already saved)
Skipping patient 11/50 (already saved)
Skipping patient 12/50 (already saved)
Skipping patient 13/50 (already saved)
Skipping patient 14/50 (already saved)
Skipping patient 15/50 (already saved)
Skipping patient 16/50 (already saved)
Skipping patient 17/50 (already saved)
Skipping patient 18/50 (already saved)
Skipping patient 19/50 (already saved)
Skipping patient 20/50 (already saved)
Skipping patient 21/50 (already saved)
Skipping patient 22/50 (already saved)
Skipping patient 23/50 (already saved)
Skipping patient 24/50 (already saved)
Skipping patient 25/50 (already saved)
Skipping pat

In [154]:
with open(ATOMIC_CLAIMS_PATH, "r") as f:
    all_atomic_claims = json.load(f)

print("Patients:", len(all_atomic_claims))

total_claims = sum(
    item["claim_count"]
    for item in all_atomic_claims.values()
)

print("Total atomic claims:", total_claims)

print(
    "Min claims per patient:",
    min(
        item["claim_count"]
        for item in all_atomic_claims.values()
    )
)

print(
    "Max claims per patient:",
    max(
        item["claim_count"]
        for item in all_atomic_claims.values()
    )
)

Patients: 50
Total atomic claims: 4896
Min claims per patient: 52
Max claims per patient: 140


In [155]:
section_chunk_embeddings = bge_model.encode(
    section_chunks["chunk_text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Chunks:", len(section_chunks))
print("Embedding shape:", section_chunk_embeddings.shape)

Batches:   0%|          | 0/87 [00:00<?, ?it/s]

Chunks: 2771
Embedding shape: (2771, 768)


In [156]:
from collections import defaultdict
import numpy as np

person_id = TEST_PATIENT_ID

patient_claims = all_atomic_claims[person_id]["claims"]

claims_by_source = defaultdict(list)

for item in patient_claims:
    claims_by_source[item["source_id"]].append(item)

test_source_id = sorted(claims_by_source.keys())[0]
source_group = claims_by_source[test_source_id]

source_sentence = source_group[0]["source_sentence"]

patient_mask = (
    section_chunks["person_id"].astype(str)
    == str(person_id)
)

patient_evidence = section_chunks[
    patient_mask
].reset_index()

patient_embeddings = section_chunk_embeddings[
    patient_mask.to_numpy()
]

retrieval_query = (
    f"Context: {source_sentence}\n\n"
    "Claims:\n"
    + "\n".join(
        f"- {item['claim']}"
        for item in source_group
    )
)

query_embedding = bge_model.encode(
    [retrieval_query],
    normalize_embeddings=True,
    show_progress_bar=False
)[0]

similarities = (
    patient_embeddings
    @ query_embedding
)

top_indices = np.argsort(
    similarities
)[::-1][:10]

retrieved_evidence = []

for rank, idx in enumerate(top_indices, start=1):
    row = patient_evidence.iloc[idx]

    retrieved_evidence.append({
        "rank": rank,
        "chunk_id": row["chunk_id"],
        "creation_timestamp": str(
            row["creation_timestamp"]
        ),
        "section_name": row["section_name"],
        "text": row["chunk_text"],
        "similarity": float(
            similarities[idx]
        ),
    })

print("Source ID:", test_source_id)
print("Source sentence:", source_sentence)

print("\nClaims:")
for item in source_group:
    print("-", item["claim"])

print("\nTop 10 evidence:")
for item in retrieved_evidence:
    print(
        f"\nRank {item['rank']} | "
        f"{item['similarity']:.3f} | "
        f"{item['section_name']}"
    )
    print(item["text"][:500])

Source ID: 1
Source sentence: **01/01/2026:** Allan Victor Robinson, 67-year-old man with hypertension and osteoarthritis, no known drug allergies and no regular medications documented, presented to A&E with acute confusion after a minor fall.

Claims:
- The patient was 67 years old.
- The patient was male.
- The patient had hypertension.
- The patient had osteoarthritis.
- The patient had no known drug allergies.
- No regular medications were documented.
- The patient presented to A&E.
- The patient had acute confusion.
- The presentation followed a minor fall.

Top 10 evidence:

Rank 1 | 0.840 | Unsectioned
- Patient: Allan Victor R obinson, 67-year-old male, NHS number 680034032. 
 - Date of event: 01/01/26, time: 28:30.
 - Event conducted by: Nurse Kay Claire Williamson. 
 - Details of event: Initial blood draws completed to investigate potential intracranial injury; patient prepared for CT head scan. 
 - Blood samples sent for: FNC, electrolytes, coagulation profile. 
 - Patient e

In [157]:
def retrieve_evidence_for_source_group(
    source_group,
    patient_evidence,
    patient_embeddings,
    top_k=10
):
    source_sentence = source_group[0]["source_sentence"]

    retrieval_query = (
        f"Context: {source_sentence}\n\n"
        "Claims:\n"
        + "\n".join(
            f"- {item['claim']}"
            for item in source_group
        )
    )

    query_embedding = bge_model.encode(
        [retrieval_query],
        normalize_embeddings=True,
        show_progress_bar=False
    )[0]

    similarities = (
        patient_embeddings
        @ query_embedding
    )

    top_indices = np.argsort(
        similarities
    )[::-1][:top_k]

    retrieved = []

    for rank, idx in enumerate(
        top_indices,
        start=1
    ):
        row = patient_evidence.iloc[idx]

        retrieved.append({
            "rank": rank,
            "chunk_id": row["chunk_id"],
            "creation_timestamp": str(
                row["creation_timestamp"]
            ),
            "section_name": row["section_name"],
            "text": row["chunk_text"],
            "similarity": float(
                similarities[idx]
            ),
        })

    return retrieved

In [158]:
test_source_retrievals = {}

for source_id, source_group in claims_by_source.items():

    retrieved = retrieve_evidence_for_source_group(
        source_group=source_group,
        patient_evidence=patient_evidence,
        patient_embeddings=patient_embeddings
    )

    test_source_retrievals[source_id] = {
        "source_id": source_id,
        "source_sentence": source_group[0]["source_sentence"],
        "claims": source_group,
        "retrieved_evidence": retrieved
    }

print("Source groups:", len(test_source_retrievals))
print(
    "Total claims:",
    sum(
        len(item["claims"])
        for item in test_source_retrievals.values()
    )
)

Source groups: 23
Total claims: 89


In [163]:
GROUP_VERIFICATION_PROMPT = """
You are verifying atomic clinical claims against evidence retrieved
from the patient's original clinical record.

For every claim, assign exactly one label:

SUPPORTED:
The original clinical evidence supports the full factual meaning
of the claim.

PARTIALLY_SUPPORTED:
Some part of the claim is supported, but another factual component
is missing, uncertain, or contradicted.

UNSUPPORTED:
The original clinical evidence does not support the claim or
explicitly contradicts it.

RULES:

1. Use only the ORIGINAL CLINICAL EVIDENCE provided.
2. Do not use outside medical knowledge.
3. Do not make unsupported inferences.
4. Semantic equivalence is allowed; wording does not need to match exactly.
5. Retrieval similarity scores are not proof of support.
6. Explicit contradiction in the relevant clinical context means UNSUPPORTED.
7. If the full atomic claim is supported, label it SUPPORTED.

8. Respect temporal context. Clinical states may change over time.
   Earlier or later findings do not automatically contradict a claim
   supported at the relevant time.

9. Use SUMMARY CONTEXT only to identify the intended clinical event,
   time point, or setting.

10. SUMMARY CONTEXT is NOT evidence and must never itself be used
    to support a claim.

11. Base every label and explanation exclusively on the original
    clinical evidence.

12. When SUMMARY CONTEXT specifies a particular time or event,
    judge the claim within that scope.

13. Findings from other time points are not contradictions when the
    patient's clinical state could have changed.

Return JSON in exactly this structure:

{
    "results": [
        {
            "claim_id": 1,
            "label": "SUPPORTED",
            "reason": "Brief explanation.",
            "supporting_evidence_ranks": [1, 3]
        }
    ]
}

Return exactly one result for every claim_id supplied.
"""


def verify_source_group(
    source_group,
    retrieved_evidence
):
    source_sentence = source_group[0]["source_sentence"]

    claims_text = "\n".join(
        f"{i}. {item['claim']}"
        for i, item in enumerate(
            source_group,
            start=1
        )
    )

    evidence_text = "\n\n".join(
        (
            f"EVIDENCE {item['rank']}:\n"
            f"Date: {item['creation_timestamp']}\n"
            f"Section: {item['section_name']}\n"
            f"{item['text']}"
        )
        for item in retrieved_evidence
    )

    user_content = (
        f"SUMMARY CONTEXT:\n{source_sentence}\n\n"
        f"ATOMIC CLAIMS:\n{claims_text}\n\n"
        f"ORIGINAL CLINICAL EVIDENCE:\n"
        f"{evidence_text}"
    )

    response = openai_client.chat.completions.create(
        model=EVALUATOR_MODEL,
        messages=[
            {
                "role": "system",
                "content": GROUP_VERIFICATION_PROMPT
            },
            {
                "role": "user",
                "content": user_content
            }
        ],
        response_format={
            "type": "json_object"
        }
    )

    return json.loads(
        response.choices[0].message.content
    )

In [164]:
test_group = test_source_retrievals[1]

test_group_verification = verify_source_group(
    source_group=test_group["claims"],
    retrieved_evidence=test_group["retrieved_evidence"]
)

for result in test_group_verification["results"]:
    claim = test_group["claims"][
        result["claim_id"] - 1
    ]["claim"]

    print(
        f"\n{result['claim_id']}. {claim}"
    )
    print("Label:", result["label"])
    print("Reason:", result["reason"])
    print(
        "Evidence:",
        result["supporting_evidence_ranks"]
    )


1. The patient was 67 years old.
Label: SUPPORTED
Reason: The record repeatedly identifies the patient as 67 years old.
Evidence: [1, 3, 5, 6, 7, 10]

2. The patient was male.
Label: SUPPORTED
Reason: The record states the patient is male.
Evidence: [1, 3, 5, 6, 10]

3. The patient had hypertension.
Label: SUPPORTED
Reason: Past medical history includes HTN/hypertension.
Evidence: [1, 3, 8, 10]

4. The patient had osteoarthritis.
Label: SUPPORTED
Reason: Past medical history includes osteoarthritis/OA.
Evidence: [1, 3, 10]

5. The patient had no known drug allergies.
Label: SUPPORTED
Reason: The record explicitly says there were no allergies / no known drug allergies.
Evidence: [3, 10]

6. No regular medications were documented.
Label: SUPPORTED
Reason: The record explicitly states there were no meds / no regular medications documented.
Evidence: [3, 8]

7. The patient presented to A&E.
Label: SUPPORTED
Reason: The patient arrived/presented to A&E in the record.
Evidence: [3]

8. The 

In [165]:
test_patient_verification = []

for source_id, source_data in test_source_retrievals.items():

    verification = verify_source_group(
        source_group=source_data["claims"],
        retrieved_evidence=source_data["retrieved_evidence"]
    )

    for result in verification["results"]:

        claim_item = source_data["claims"][
            result["claim_id"] - 1
        ]

        test_patient_verification.append({
            "source_id": source_id,
            "claim": claim_item["claim"],
            "source_sentence": claim_item["source_sentence"],
            "label": result["label"],
            "reason": result["reason"],
            "supporting_evidence_ranks": result[
                "supporting_evidence_ranks"
            ],
            "retrieved_evidence": source_data[
                "retrieved_evidence"
            ],
        })

print(
    "Verified claims:",
    len(test_patient_verification)
)

Verified claims: 89


In [166]:
from collections import Counter

label_counts = Counter(
    item["label"]
    for item in test_patient_verification
)

print(label_counts)

Counter({'SUPPORTED': 89})


In [167]:
total_claims = sum(
    patient_data["claim_count"]
    for patient_data in all_atomic_claims.values()
)

total_source_groups = sum(
    len({
        item["source_id"]
        for item in patient_data["claims"]
    })
    for patient_data in all_atomic_claims.values()
)

print("Patients:", len(all_atomic_claims))
print("Total atomic claims:", total_claims)
print("Total source_id groups:", total_source_groups)
print("GPT verification calls needed:", total_source_groups)
print(
    "Average claims per call:",
    round(total_claims / total_source_groups, 2)
)

Patients: 50
Total atomic claims: 4896
Total source_id groups: 1148
GPT verification calls needed: 1148
Average claims per call: 4.26


In [168]:
VERIFICATION_RESULTS_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "rag_verification_results.json"
)

if VERIFICATION_RESULTS_PATH.exists():
    with open(VERIFICATION_RESULTS_PATH, "r") as f:
        all_verification_results = json.load(f)
else:
    all_verification_results = {}

print(
    "Patients already verified:",
    len(all_verification_results)
)

Patients already verified: 0


In [169]:
BGE_RETRIEVAL_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "rag_verification_bge_retrievals.json"
)

if BGE_RETRIEVAL_PATH.exists():
    with open(BGE_RETRIEVAL_PATH, "r") as f:
        all_bge_retrievals = json.load(f)
else:
    all_bge_retrievals = {}

print(
    "Patients already retrieved:",
    len(all_bge_retrievals)
)

Patients already retrieved: 0


In [173]:
def retrieve_evidence_for_source_group(
    source_group,
    patient_evidence,
    patient_embeddings,
    top_k=10
):
    source_sentence = source_group[0]["source_sentence"]

    retrieval_query = (
        f"Context: {source_sentence}\n\n"
        "Claims:\n"
        + "\n".join(
            f"- {item['claim']}"
            for item in source_group
        )
    )

    query_embedding = bge_model.encode(
        [retrieval_query],
        normalize_embeddings=True,
        show_progress_bar=False
    )[0]

    similarities = (
        patient_embeddings
        @ query_embedding
    )

    top_indices = np.argsort(
        similarities
    )[::-1][:top_k]

    retrieved = []

    for rank, idx in enumerate(
        top_indices,
        start=1
    ):
        row = patient_evidence.iloc[int(idx)]

        retrieved.append({
            "rank": int(rank),
            "chunk_id": int(row["chunk_id"]),
            "creation_timestamp": str(
                row["creation_timestamp"]
            ),
            "section_name": str(
                row["section_name"]
            ),
            "text": str(
                row["chunk_text"]
            ),
            "similarity": float(
                similarities[int(idx)]
            ),
        })

    return retrieved

In [177]:
def make_json_safe(obj):

    if isinstance(obj, dict):
        return {
            str(key): make_json_safe(value)
            for key, value in obj.items()
        }

    if isinstance(obj, list):
        return [
            make_json_safe(value)
            for value in obj
        ]

    if isinstance(obj, tuple):
        return [
            make_json_safe(value)
            for value in obj
        ]

    if isinstance(obj, np.integer):
        return int(obj)

    if isinstance(obj, np.floating):
        return float(obj)

    if isinstance(obj, np.ndarray):
        return obj.tolist()

    return obj

In [178]:
all_bge_retrievals = {}

In [179]:
for patient_idx, (person_id, patient_data) in enumerate(
    all_atomic_claims.items(),
    start=1
):

    if person_id in all_bge_retrievals:
        print(
            f"Skipping patient {patient_idx}/{len(all_atomic_claims)} "
            "(already saved)"
        )
        continue

    print(
        f"\nRetrieving patient "
        f"{patient_idx}/{len(all_atomic_claims)}"
    )

    # Original clinical-note chunks for this patient
    patient_mask = (
        section_chunks["person_id"].astype(str)
        == str(person_id)
    )

    patient_evidence = (
        section_chunks[
            patient_mask
        ]
        .reset_index(drop=True)
    )

    patient_embeddings = (
        section_chunk_embeddings[
            patient_mask.to_numpy()
        ]
    )

    # Group existing atomic claims by source_id
    claims_by_source = defaultdict(list)

    for claim_id, item in enumerate(
        patient_data["claims"],
        start=1
    ):
        claims_by_source[
            item["source_id"]
        ].append({
            "claim_id": claim_id,
            **item
        })

    patient_retrievals = {}

    # BGE retrieval only — NO GPT
    for source_id, source_group in claims_by_source.items():

        retrieved_evidence = (
            retrieve_evidence_for_source_group(
                source_group=source_group,
                patient_evidence=patient_evidence,
                patient_embeddings=patient_embeddings,
                top_k=10
            )
        )

        patient_retrievals[str(int(source_id))] = {
            "source_id": source_id,
            "source_sentence": (
                source_group[0]["source_sentence"]
            ),
            "claims": source_group,
            "retrieved_evidence": retrieved_evidence
        }

    all_bge_retrievals[person_id] = {
        "person_id": person_id,
        "source_group_count": len(
            patient_retrievals
        ),
        "source_groups": patient_retrievals
    }

    # checkpoint after each patient
    with open(
        BGE_RETRIEVAL_PATH,
        "w"
    ) as f:
        json.dump(
            make_json_safe(all_bge_retrievals),
            f,
            indent=2
        )

    print(
        "Source groups saved:",
        len(patient_retrievals)
    )
    print("Checkpoint saved.")


Retrieving patient 1/50
Source groups saved: 15
Checkpoint saved.

Retrieving patient 2/50
Source groups saved: 23
Checkpoint saved.

Retrieving patient 3/50
Source groups saved: 29
Checkpoint saved.

Retrieving patient 4/50
Source groups saved: 19
Checkpoint saved.

Retrieving patient 5/50
Source groups saved: 20
Checkpoint saved.

Retrieving patient 6/50
Source groups saved: 23
Checkpoint saved.

Retrieving patient 7/50
Source groups saved: 21
Checkpoint saved.

Retrieving patient 8/50
Source groups saved: 30
Checkpoint saved.

Retrieving patient 9/50
Source groups saved: 30
Checkpoint saved.

Retrieving patient 10/50
Source groups saved: 22
Checkpoint saved.

Retrieving patient 11/50
Source groups saved: 13
Checkpoint saved.

Retrieving patient 12/50
Source groups saved: 37
Checkpoint saved.

Retrieving patient 13/50
Source groups saved: 26
Checkpoint saved.

Retrieving patient 14/50
Source groups saved: 23
Checkpoint saved.

Retrieving patient 15/50
Source groups saved: 36
Checkpo

In [180]:
total_source_groups_saved = sum(
    patient_data["source_group_count"]
    for patient_data in all_bge_retrievals.values()
)

print(
    "Patients saved:",
    len(all_bge_retrievals)
)

print(
    "Source groups saved:",
    total_source_groups_saved
)

Patients saved: 50
Source groups saved: 1148


In [ ]:
# --------------------------------------------------
# Output path
# --------------------------------------------------

VERIFICATION_RESULTS_PATH = (
    PROJECT_ROOT
    / "data"
    / "results"
    /"rag_verification"
    / "rag_verification_results.json"
)

# --------------------------------------------------
# Load existing checkpoint
# --------------------------------------------------

if VERIFICATION_RESULTS_PATH.exists():
    with open(VERIFICATION_RESULTS_PATH, "r") as f:
        all_verification_results = json.load(f)
else:
    all_verification_results = {}

print(
    "Patients currently in verification file:",
    len(all_verification_results)
)

# --------------------------------------------------
# SAFETY:
# First run only 3 NEW patients.
#
# After we inspect them, change this to:
# MAX_NEW_PATIENTS = None
# --------------------------------------------------

MAX_NEW_PATIENTS = 3

new_patients_completed = 0


# --------------------------------------------------
# Verification loop
# --------------------------------------------------

for patient_idx, (
    person_id,
    patient_retrieval_data
) in enumerate(
    all_bge_retrievals.items(),
    start=1
):

    person_id = str(person_id)

    # Create patient checkpoint structure if needed
    if person_id not in all_verification_results:
        all_verification_results[person_id] = {
            "person_id": person_id,
            "source_group_count": int(
                patient_retrieval_data[
                    "source_group_count"
                ]
            ),
            "source_groups": {}
        }

    patient_checkpoint = (
        all_verification_results[person_id]
    )

    source_groups = (
        patient_retrieval_data["source_groups"]
    )

    completed_before = len(
        patient_checkpoint["source_groups"]
    )

    # If patient already completely verified, skip
    if completed_before == len(source_groups):
        print(
            f"Skipping patient "
            f"{patient_idx}/{len(all_bge_retrievals)} "
            "(already complete)"
        )
        continue

    print(
        f"\nVerifying patient "
        f"{patient_idx}/{len(all_bge_retrievals)}"
    )
    print("Person ID:", person_id)
    print(
        "Source groups:",
        len(source_groups)
    )
    print(
        "Already verified:",
        completed_before
    )

    # --------------------------------------------------
    # Verify each source_id group
    # --------------------------------------------------

    for group_idx, (
        source_id,
        source_data
    ) in enumerate(
        source_groups.items(),
        start=1
    ):

        source_id = str(source_id)

        # Skip already checkpointed source groups
        if source_id in patient_checkpoint[
            "source_groups"
        ]:
            continue

        print(
            f"  Group "
            f"{group_idx}/{len(source_groups)} "
            f"(source_id={source_id})"
        )

        source_group = source_data["claims"]

        retrieved_evidence = (
            source_data["retrieved_evidence"]
        )

        # ------------------------------------------
        # One GPT-5.4-mini call for this source group
        # ------------------------------------------

        verification = verify_source_group(
            source_group=source_group,
            retrieved_evidence=retrieved_evidence
        )

        results = verification["results"]

        # ------------------------------------------
        # Validate response
        # ------------------------------------------

        if len(results) != len(source_group):
            raise ValueError(
                f"Result count mismatch for "
                f"patient {person_id}, "
                f"source_id {source_id}: "
                f"{len(source_group)} claims, "
                f"{len(results)} results."
            )

        expected_ids = set(
            range(
                1,
                len(source_group) + 1
            )
        )

        returned_ids = {
            int(item["claim_id"])
            for item in results
        }

        if returned_ids != expected_ids:
            raise ValueError(
                f"Claim ID mismatch for "
                f"patient {person_id}, "
                f"source_id {source_id}. "
                f"Expected {expected_ids}, "
                f"got {returned_ids}."
            )

        # ------------------------------------------
        # Attach results to original claims
        # ------------------------------------------

        verified_claims = []

        for result in results:

            local_claim_id = int(
                result["claim_id"]
            )

            original_claim = source_group[
                local_claim_id - 1
            ]

            verified_claims.append({
                "claim_id": int(
                    original_claim["claim_id"]
                ),
                "claim": original_claim["claim"],
                "source_id": int(
                    original_claim["source_id"]
                ),
                "source_sentence": (
                    original_claim[
                        "source_sentence"
                    ]
                ),
                "label": result["label"],
                "reason": result["reason"],
                "supporting_evidence_ranks": (
                    result[
                        "supporting_evidence_ranks"
                    ]
                )
            })

        # ------------------------------------------
        # Save this source group immediately
        # ------------------------------------------

        patient_checkpoint[
            "source_groups"
        ][source_id] = {
            "source_id": int(source_id),
            "source_sentence": (
                source_data["source_sentence"]
            ),
            "verified_claims": verified_claims
        }

        with open(
            VERIFICATION_RESULTS_PATH,
            "w"
        ) as f:
            json.dump(
                all_verification_results,
                f,
                indent=2
            )

    # --------------------------------------------------
    # Patient finished
    # --------------------------------------------------

    patient_labels = []

    for group_data in patient_checkpoint[
        "source_groups"
    ].values():

        patient_labels.extend(
            claim["label"]
            for claim in group_data[
                "verified_claims"
            ]
        )

    print(
        "Patient complete:",
        Counter(patient_labels)
    )

    new_patients_completed += 1

    if (
        MAX_NEW_PATIENTS is not None
        and
        new_patients_completed >= MAX_NEW_PATIENTS
    ):
        print(
            "\nStopped after",
            MAX_NEW_PATIENTS,
            "new patients for inspection."
        )
        break